Instalação e Importações

In [2]:
import duckdb
import boto3
import pandas as pd
from datetime import datetime, timedelta
from tqdm.notebook import tqdm # Barra de progresso visual
import warnings

# Suprime avisos do pandas para manter o notebook limpo
warnings.filterwarnings('ignore')

Configurações e Mapeamento

In [ ]:
# --- CONFIGURAÇÕES ---
BUCKET = "transdesk-develop-bronze"
DATA_INICIO = "2026-01-01"
DATA_FIM = "2026-06-30"

# Mapeamento: Esquerda (Nome no SQL) -> Direita (Pasta real no S3)
SCHEMAS = {
    'silver': 'vilesoft',   
    'stcoop': 'stcoop',
    'viavante': 'viavante',
    'tag': 'tag'
}

# Lista de tabelas necessárias para baixar
TABELAS = [
    "insurance_registration", "insurance_reg_set", "cliente", "catalogo", 
    "representante", "insurance_status", "INSURANCE_REG_SET_COVERAGE", 
    "INSURANCE_VEHICLE", "TIPO_VEICULO", "INSURANCE_REG_SET_COV_TRAILER", 
    "INSURANCE_TRAILER", "vendedor", "price_list_benefits", "type_category", 
    "category", "benefits", "web_user"
]

Inicialização do DuckDB e Credenciais AWS

In [4]:
# Inicia banco em memória
con = duckdb.connect(database=':memory:')
con.execute("INSTALL httpfs; LOAD httpfs;") 

# Pega credenciais da sua sessão AWS local
session = boto3.Session()
creds = session.get_credentials().get_frozen_credentials()

# Configura o DuckDB para acessar o S3
con.execute(f"""
    SET s3_region='sa-east-1';
    SET s3_access_key_id='{creds.access_key}';
    SET s3_secret_access_key='{creds.secret_key}';
    SET s3_session_token='{creds.token}';
""")

# Cria os schemas vazios para o SQL não quebrar antes de carregar os dados
for sql_schema_name in SCHEMAS.keys():
    con.execute(f"CREATE SCHEMA IF NOT EXISTS {sql_schema_name};")
    
print("DuckDB configurado e schemas criados!")

DuckDB configurado e schemas criados!


Definição da Query SQL

In [5]:
QUERY_BASE = """
WITH base_unificada AS (
    -- BLOCO SILVER (Aponta para Vilesoft via mapeamento)
    SELECT 
        coalesce(iv.BOARD,it.BOARD,itt.BOARD) as "placa",
        coalesce(iv.chassi,it.chassi,itt.chassi) as "chassi",
        cast(coalesce(irs.DATE_INITAL_EFFECT,date_add('day', -364, irs.DATE_FINAL_EFFECT)) as date) as "inicio_vig",
        'Segtruck' as empresa
    from silver.insurance_registration ir
        left outer join silver.insurance_reg_set irs on irs.parent = ir.id
        left outer join silver.cliente clie on clie.codigo = ir.CUSTOMER_ID
        left outer join silver.catalogo cat on cat.cnpj_cpf = clie.cnpj_cpf
        left outer join silver.representante r on r.codigo = irs.id_unity
        left outer join silver.catalogo cata on cata.cnpj_cpf = r.cnpj_cpf
        left outer join silver.insurance_status iss on iss.id = irs.id_status
        left outer join silver.INSURANCE_REG_SET_COVERAGE irsc ON irsc.PARENT = irs.ID
        left outer join silver.INSURANCE_VEHICLE iv on iv.ID = irsc.ID_VEHICLE
        left outer join silver.INSURANCE_REG_SET_COV_TRAILER irsct on irsct.PARENT = irsc.ID
        left outer join silver.INSURANCE_TRAILER it on it.ID = irsct.ID_TRAILER
        left outer join silver.insurance_trailer itt on itt.id = irsc.ID_TRAILER
        left outer join silver.insurance_status isss on isss.id = irsc.id_status
    where iss.id = 7
    and (coalesce(iv.BOARD,it.BOARD,itt.BOARD) is not null and coalesce(iv.chassi,it.chassi,itt.chassi) is not null)
    and isss.description = 'ATIVO'

    UNION ALL

    -- BLOCO STCOOP
    SELECT 
        coalesce(iv.BOARD,it.BOARD,itt.BOARD) as "placa",
        coalesce(iv.chassi,it.chassi,itt.chassi) as "chassi",
        cast(coalesce(irs.DATE_INITAL_EFFECT,date_add('day', -364, irs.DATE_FINAL_EFFECT)) as date) as "inicio_vig",
        'Stcoop' as empresa
    from stcoop.insurance_registration ir
        left outer join stcoop.insurance_reg_set irs on irs.parent = ir.id
        left outer join stcoop.cliente clie on clie.codigo = ir.CUSTOMER_ID
        left outer join stcoop.catalogo cat on cat.cnpj_cpf = clie.cnpj_cpf
        left outer join stcoop.representante r on r.codigo = irs.id_unity
        left outer join stcoop.catalogo cata on cata.cnpj_cpf = r.cnpj_cpf
        left outer join stcoop.insurance_status iss on iss.id = irs.id_status
        left outer join stcoop.INSURANCE_REG_SET_COVERAGE irsc ON irsc.PARENT = irs.ID
        left outer join stcoop.INSURANCE_VEHICLE iv on iv.ID = irsc.ID_VEHICLE
        left outer join stcoop.INSURANCE_REG_SET_COV_TRAILER irsct on irsct.PARENT = irsc.ID
        left outer join stcoop.INSURANCE_TRAILER it on it.ID = irsct.ID_TRAILER
        left outer join stcoop.insurance_trailer itt on itt.id = irsc.ID_TRAILER
        left outer join stcoop.insurance_status isss on isss.id = irsc.id_status
    where iss.id = 7
    and (coalesce(iv.BOARD,it.BOARD,itt.BOARD) is not null and coalesce(iv.chassi,it.chassi,itt.chassi) is not null)
    and isss.description = 'ATIVO'

    UNION ALL

    -- BLOCO VIAVANTE
    SELECT 
        coalesce(iv.BOARD,it.BOARD,itt.BOARD) as "placa",
        coalesce(iv.chassi,it.chassi,itt.chassi) as "chassi",
        cast(coalesce(irs.DATE_INITAL_EFFECT,date_add('day', -364, irs.DATE_FINAL_EFFECT)) as date) as "inicio_vig",
        'Viavante' as empresa
    from viavante.insurance_registration ir
        left outer join viavante.insurance_reg_set irs on irs.parent = ir.id
        left outer join viavante.cliente clie on clie.codigo = ir.CUSTOMER_ID
        left outer join viavante.catalogo cat on cat.cnpj_cpf = clie.cnpj_cpf
        left outer join viavante.representante r on r.codigo = irs.id_unity
        left outer join viavante.catalogo cata on cata.cnpj_cpf = r.cnpj_cpf
        left outer join viavante.insurance_status iss on iss.id = irs.id_status
        left outer join viavante.INSURANCE_REG_SET_COVERAGE irsc ON irsc.PARENT = irs.ID
        left outer join viavante.INSURANCE_VEHICLE iv on iv.ID = irsc.ID_VEHICLE
        left outer join viavante.INSURANCE_REG_SET_COV_TRAILER irsct on irsct.PARENT = irsc.ID
        left outer join viavante.INSURANCE_TRAILER it on it.ID = irsct.ID_TRAILER
        left outer join viavante.insurance_trailer itt on itt.id = irsc.ID_TRAILER
        left outer join viavante.insurance_status isss on isss.id = irsc.id_status
    where iss.id = 7
    and (coalesce(iv.BOARD,it.BOARD,itt.BOARD) is not null and coalesce(iv.chassi,it.chassi,itt.chassi) is not null)
    and isss.description = 'ATIVO'

    UNION ALL

    -- BLOCO TAG
    SELECT 
        coalesce(iv.BOARD,it.BOARD,itt.BOARD) as "placa",
        coalesce(iv.chassi,it.chassi,itt.chassi) as "chassi",
        cast(coalesce(irs.DATE_INITAL_EFFECT,date_add('day', -364, irs.DATE_FINAL_EFFECT)) as date) as "inicio_vig",
        'Tag' as empresa
    from tag.insurance_registration ir
        left outer join tag.insurance_reg_set irs on irs.parent = ir.id
        left outer join tag.cliente clie on clie.codigo = ir.CUSTOMER_ID
        left outer join tag.catalogo cat on cat.cnpj_cpf = clie.cnpj_cpf
        left outer join tag.representante r on r.codigo = irs.id_unity
        left outer join tag.catalogo cata on cata.cnpj_cpf = r.cnpj_cpf
        left outer join tag.insurance_status iss on iss.id = irs.id_status
        left outer join tag.INSURANCE_REG_SET_COVERAGE irsc ON irsc.PARENT = irs.ID
        left outer join tag.INSURANCE_VEHICLE iv on iv.ID = irsc.ID_VEHICLE
        left outer join tag.INSURANCE_REG_SET_COV_TRAILER irsct on irsct.PARENT = irsc.ID
        left outer join tag.INSURANCE_TRAILER it on it.ID = irsct.ID_TRAILER
        left outer join tag.insurance_trailer itt on itt.id = irsc.ID_TRAILER
        left outer join tag.insurance_status isss on isss.id = irsc.id_status
    where iss.id = 7
    and (coalesce(iv.BOARD,it.BOARD,itt.BOARD) is not null and coalesce(iv.chassi,it.chassi,itt.chassi) is not null)
    and isss.description = 'ATIVO'
)
SELECT * FROM base_unificada
"""

Loop Principal

In [7]:
resultados = []

# Gerador de datas
def daterange(start_date, end_date):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    for n in range(int((end - start).days) + 1):
        yield (start + timedelta(n)).strftime("%Y-%m-%d")

# Cria lista de datas
lista_datas = list(daterange(DATA_INICIO, DATA_FIM))

print(f"Iniciando loop de processamento para {len(lista_datas)} dias...")

# Loop com barra de progresso (tqdm)
for dia_x in tqdm(lista_datas, desc="Processando Dias"):
    
    # 1. Montar Views (Ligação SQL <-> S3)
    for sql_name, s3_folder in SCHEMAS.items():
        for tb in TABELAS:
            caminho = f"s3://{BUCKET}/{s3_folder}/{tb}/landing_date={dia_x}/*.parquet"
            try:
                # Tenta criar view com arquivos reais
                con.execute(f"""
                    CREATE OR REPLACE VIEW {sql_name}.{tb} AS 
                    SELECT * FROM read_parquet('{caminho}', hive_partitioning=1, union_by_name=True)
                """)
            except:
                # Se falhar (arquivo não existe), cria view vazia
                con.execute(f"CREATE OR REPLACE VIEW {sql_name}.{tb} AS SELECT * FROM {sql_name}.{tb} WHERE 1=0")

    try:
        # 2. Executar SQL
        df_dia = con.execute(QUERY_BASE).df()
        
        if df_dia.empty:
            count_ativos = 0
        else:
            # 3. Processamento Pandas
            df_dia['inicio_vig'] = pd.to_datetime(df_dia['inicio_vig'], errors='coerce')
            
            # Ordenar por vigência (mais recente primeiro)
            df_dia = df_dia.sort_values(by='inicio_vig', ascending=False)
            
            # Remover duplicatas de chassi
            df_limpo = df_dia.drop_duplicates(subset=['chassi'], keep='first')
            
            # Contar
            count_ativos = len(df_limpo)
        
        resultados.append({'data': dia_x, 'ativos': count_ativos})

    except Exception as e:
        print(f"ERRO no dia {dia_x}: {e}")
        resultados.append({'data': dia_x, 'ativos': 0, 'erro': str(e)})

print("Processamento concluído.")

Iniciando loop de processamento para 181 dias...


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html